In [41]:
#import tranfer_instructions.csv 
import pandas as pd

# Import transfer instructions CSV
df_transfers = pd.read_csv('data/transfer_instructions.csv')

# Display basic info
print(f"Total transfers: {len(df_transfers)}")
print(f"\nColumns: {list(df_transfers.columns)}")
print(f"\nFirst few rows:")
print(df_transfers.head(10))

# Summary statistics
print(f"\nSummary:")
print(f"  Unique source plates: {df_transfers['Source_Plate'].nunique()}")
print(f"  Unique destination plates: {df_transfers['Dest_Plate'].nunique()}")
print(f"  Unique destination wells: {df_transfers['Dest_Well'].nunique()}")
print(f"  Total volume to transfer: {df_transfers['Transfer_Vol'].sum():.2f} µL")
print(f"  Average transfer volume: {df_transfers['Transfer_Vol'].mean():.2f} µL")

Total transfers: 1932

Columns: ['Source_Plate', 'Source_Well', 'Dest_Plate', 'Dest_Well', 'Transfer_Vol']

First few rows:
  Source_Plate Source_Well Dest_Plate Dest_Well  Transfer_Vol
0           s1          A1     dest_1        A1         99.65
1           s1          A2     dest_1        A1         93.54
2           s1          A3     dest_1        A1         94.56
3           s1          A5     dest_1        A1         98.00
4           s1          A6     dest_1        A1         84.92
5           s4          C4     dest_1        A1          6.74
6      s_water        none     dest_1        A1        312.59
7           s1          A1     dest_1       A10         92.31
8           s1          A2     dest_1       A10         69.88
9           s1          A3     dest_1       A10         85.02

Summary:
  Unique source plates: 3
  Unique destination plates: 3
  Unique destination wells: 276
  Total volume to transfer: 218040.00 µL
  Average transfer volume: 112.86 µL


In [42]:
## find number of rows with Transfer_Vol <10 and list those rows
df_10ul = df_transfers[df_transfers['Transfer_Vol'] < 10]
print(df_10ul)
#convert to csv
df_10ul.to_csv('data/df_10ul.csv', index=False)
#list all source plates and well pairs in df_10ul

     Source_Plate Source_Well Dest_Plate Dest_Well  Transfer_Vol
5              s4          C4     dest_1        A1          6.74
558            s4          C4     dest_1        G5          7.83
572            s4          C4     dest_1        G7          0.29
670            s4          C4     dest_1        H9          3.38
754            s4          C4     dest_2        I9          7.83
1279           s4          C4     dest_2       N33          6.74
1587           s4          C6     dest_3       N50          1.31
1804           s4          C4     dest_3       N81          0.29


In [43]:
#list all source plates and well pairs in df_10ul
df_10ul[['Source_Plate', 'Source_Well']]
#list unique source plates in df_10ul
df_10ul['Source_Plate'].unique()




array(['s4'], dtype=object)

In [44]:
# return max value of Transfer_Vol in df_transfers for source plate 1 well 1 pair; source plate 2 and well pair 
# Group by Source_Plate and Source_Well, then find max Transfer_Vol for each pair
max_transfers = df_transfers.groupby(['Source_Plate', 'Source_Well'])['Transfer_Vol'].max().reset_index()
max_transfers.columns = ['Source_Plate', 'Source_Well', 'Max_Transfer_Vol']

# Sort by Source_Plate and Source_Well for better readability
max_transfers = max_transfers.sort_values(['Source_Plate', 'Source_Well']).reset_index(drop=True)

print("Maximum Transfer Volumes for each Source Plate and Well pair:")
print("=" * 70)
print(max_transfers.to_string(index=False))

print(f"\n\nSummary:")
print(f"  Total unique source plate/well pairs: {len(max_transfers)}")
print(f"  Overall maximum transfer volume: {max_transfers['Max_Transfer_Vol'].max():.2f} µL")
print(f"  Overall minimum maximum transfer volume: {max_transfers['Max_Transfer_Vol'].min():.2f} µL")
print(f"  Average of maximum transfer volumes: {max_transfers['Max_Transfer_Vol'].mean():.2f} µL")

# Optionally save to CSV
max_transfers.to_csv('data/max_transfer_volumes_by_source_well.csv', index=False)
print(f"\n  Results saved to: data/max_transfer_volumes_by_source_well.csv")



Maximum Transfer Volumes for each Source Plate and Well pair:
Source_Plate Source_Well  Max_Transfer_Vol
          s1          A1             99.72
          s1          A2             99.82
          s1          A3            166.60
          s1          A4            248.25
          s1          A5            166.35
          s1          A6             99.77
          s1          B1            249.07
          s1          B2            166.49
          s1          B3            166.35
          s1          B4            246.80
          s1          B5             99.90
          s1          B6             99.72
          s1          C1             96.68
          s1          C2            166.66
          s1          C3            156.67
          s1          C4            248.25
          s4          C1            246.27
          s4          C2             99.81
          s4          C3            289.49
          s4          C4             79.25
          s4          C5           

In [46]:
# for each Source_Plate Source_Well pair, find the total transfer volume for each unique pair. example S1 A1 pipetted total how much volume in the dataframe

# Group by Source_Plate and Source_Well, then sum Transfer_Vol for each pair
total_transfers = df_transfers.groupby(['Source_Plate', 'Source_Well'])['Transfer_Vol'].sum().reset_index()
total_transfers.columns = ['Source_Plate', 'Source_Well', 'Total_Transfer_Vol']

# Sort by Source_Plate and Source_Well for better readability
total_transfers = total_transfers.sort_values(['Source_Plate', 'Source_Well']).reset_index(drop=True)

print("Total Transfer Volumes for each Source Plate and Well pair:")
print("=" * 80)
print(total_transfers.to_string(index=False))

print(f"\n\nSummary:")
print(f"  Total unique source plate/well pairs: {len(total_transfers)}")
print(f"  Highest total volume from single source: {total_transfers['Total_Transfer_Vol'].max():.2f} µL")
print(f"  Lowest total volume from single source: {total_transfers['Total_Transfer_Vol'].min():.2f} µL")
print(f"  Average total volume per source: {total_transfers['Total_Transfer_Vol'].mean():.2f} µL")
print(f"  Grand total across all sources: {total_transfers['Total_Transfer_Vol'].sum():.2f} µL")

# Show top 10 sources by total volume
print(f"\n\nTop 10 Source Wells by Total Volume:")
print("=" * 80)
top_10 = total_transfers.nlargest(10, 'Total_Transfer_Vol')
for idx, row in top_10.iterrows():
    print(f"  {row['Source_Plate']}:{row['Source_Well']:4s} -> {row['Total_Transfer_Vol']:10.2f} µL")

# Optionally save to CSV
total_transfers.to_csv('data/total_transfer_volumes_by_source_well.csv', index=False)
print(f"\n  Results saved to: data/total_transfer_volumes_by_source_well.csv")

Total Transfer Volumes for each Source Plate and Well pair:
Source_Plate Source_Well  Total_Transfer_Vol
          s1          A1             8895.56
          s1          A2             8896.05
          s1          A3             8896.29
          s1          A4             8895.69
          s1          A5             8899.20
          s1          A6             8896.38
          s1          B1             8892.80
          s1          B2             8891.44
          s1          B3             8899.37
          s1          B4             8894.98
          s1          B5             4901.55
          s1          B6             4202.05
          s1          C1             3163.07
          s1          C2             4996.61
          s1          C3             2701.14
          s1          C4             2930.78
          s4          C1             4562.75
          s4          C2             2165.19
          s4          C3             2630.40
          s4          C4              54

In [47]:
# water volumes in each destination well

# Filter for water transfers (from s_water source)
water_transfers = df_transfers[df_transfers['Source_Plate'] == 's_water'].copy()

# Group by destination plate and well to get total water volume per well
water_volumes = water_transfers.groupby(['Dest_Plate', 'Dest_Well'])['Transfer_Vol'].sum().reset_index()
water_volumes.columns = ['Dest_Plate', 'Dest_Well', 'Water_Volume_µL']

# Sort by destination plate and well for better readability
water_volumes = water_volumes.sort_values(['Dest_Plate', 'Dest_Well']).reset_index(drop=True)

print("Water Volumes for each Destination Well:")
print("=" * 80)
print(water_volumes.to_string(index=False))

print(f"\n\nSummary:")
print(f"  Total destination wells with water: {len(water_volumes)}")
print(f"  Highest water volume in a single well: {water_volumes['Water_Volume_µL'].max():.2f} µL")
print(f"  Lowest water volume in a single well: {water_volumes['Water_Volume_µL'].min():.2f} µL")
print(f"  Average water volume per well: {water_volumes['Water_Volume_µL'].mean():.2f} µL")
print(f"  Total water volume across all wells: {water_volumes['Water_Volume_µL'].sum():.2f} µL")

# Show wells with very low water volumes (potential issues)
low_water = water_volumes[water_volumes['Water_Volume_µL'] < 50]
if len(low_water) > 0:
    print(f"\n\n⚠️  Wells with low water volume (< 50 µL): {len(low_water)}")
    print("=" * 80)
    print(low_water.to_string(index=False))
else:
    print(f"\n✅ All wells have adequate water volume (>= 50 µL)")

# Show wells with very high water volumes
high_water = water_volumes[water_volumes['Water_Volume_µL'] > 800]
if len(high_water) > 0:
    print(f"\n\n📊 Wells with high water volume (> 800 µL): {len(high_water)}")
    print("=" * 80)
    print(high_water.head(10).to_string(index=False))
    if len(high_water) > 10:
        print(f"  ... and {len(high_water) - 10} more wells")

# Optionally save to CSV
water_volumes.to_csv('data/water_volumes_by_destination_well.csv', index=False)
print(f"\n  Results saved to: data/water_volumes_by_destination_well.csv")

Water Volumes for each Destination Well:
Dest_Plate Dest_Well  Water_Volume_µL
    dest_1        A1           312.59
    dest_1       A10           223.73
    dest_1       A11           110.65
    dest_1       A12           285.92
    dest_1        A2           340.49
    dest_1        A3           227.78
    dest_1        A4           416.47
    dest_1        A5           186.97
    dest_1        A6           194.37
    dest_1        A7           360.77
    dest_1        A8           342.54
    dest_1        A9           170.50
    dest_1        B1           309.79
    dest_1       B10           232.19
    dest_1       B11           308.37
    dest_1       B12           177.46
    dest_1        B2           373.02
    dest_1        B3           326.47
    dest_1        B4           365.51
    dest_1        B5           418.89
    dest_1        B6           320.80
    dest_1        B7           399.87
    dest_1        B8           440.94
    dest_1        B9           264.88
    dest_